In [1]:
import polars as pl
import numpy as np

def empirical_pricer(
    underlying_df: pl.DataFrame,
    current_price: float,
    days_to_expiry: float,
    strikes: list[int] = [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500],
    simulations: int = 5000,
    ticks_per_day: int = 10000 
) -> dict[int, float]:
    """
    Prices options empirically by bootstrapping historical log returns.
    Expects underlying_df to have a 'mid_price' column sorted by time.
    """
    # 1. Extract empirical distribution of tick-to-tick log returns
    df = underlying_df.with_columns(
        (pl.col("mid_price") / pl.col("mid_price").shift(1)).log().alias("log_return")
    ).drop_nulls()
    
    historical_returns = df["log_return"].to_numpy()
    
    # 2. Calculate remaining simulation steps
    steps_remaining = int(days_to_expiry * ticks_per_day)
    if steps_remaining <= 0:
        return {k: max(current_price - k, 0) for k in strikes}
        
    # 3. Bootstrap future paths (sample with replacement)
    # Note: size is (simulations, steps_remaining). Adjust 'simulations' if RAM is constrained.
    sampled_returns = np.random.choice(
        historical_returns, 
        size=(simulations, steps_remaining), 
        replace=True
    )
    
    # 4. Calculate terminal prices
    # X_T = X_0 * exp(sum(log_returns))
    cumulative_log_returns = np.sum(sampled_returns, axis=1)
    terminal_prices = current_price * np.exp(cumulative_log_returns)
    
    # 5. Calculate empirical expected payoff for each strike
    fair_values = {}
    for k in strikes:
        payoffs = np.maximum(terminal_prices - k, 0)
        # Interest rate is assumed to be 0 for the competition
        fair_values[k] = np.mean(payoffs) 
        
    return fair_values

# ==========================================
# Example Usage 
# ==========================================

# Generate synthetic historical data representing VELVETFRUIT_EXTRACT
np.random.seed(42)
n_historical_ticks = 10000 
synthetic_returns = np.random.normal(loc=0.000002, scale=0.0004, size=n_historical_ticks)
synthetic_prices = 5000 * np.exp(np.cumsum(synthetic_returns))

underlying_data = pl.DataFrame({
    "timestamp": np.arange(n_historical_ticks),
    "mid_price": synthetic_prices
})

# State at the start of Round 3 (TTE = 5 days)
current_velvet_price = underlying_data["mid_price"].tail(1).item()

empirical_fvs = empirical_pricer(
    underlying_df=underlying_data,
    current_price=current_velvet_price,
    days_to_expiry=5.0,  
    strikes=[4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500],
    simulations=10000
)

print(f"Current VELVETFRUIT_EXTRACT price: {current_velvet_price:.2f}\n")
print("Empirical Fair Values for VEV Vouchers (TTE = 5d):")
for strike, fv in empirical_fvs.items():
    print(f"VEV_{strike}: {fv:.2f}")

Current VELVETFRUIT_EXTRACT price: 5057.61

Empirical Fair Values for VEV Vouchers (TTE = 5d):
VEV_4000: 1380.67
VEV_4500: 884.13
VEV_5000: 434.03
VEV_5100: 359.65
VEV_5200: 292.67
VEV_5300: 233.78
VEV_5400: 183.09
VEV_5500: 140.58
VEV_6000: 27.54
VEV_6500: 3.15
